# Nassau Candy Distributor Profitability EDA
This notebook reproduces the core cleaning, KPI, Pareto, and risk calculations used in the dashboard and reports.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA = Path("../data/Nassau Candy Distributor.csv")
df = pd.read_csv(DATA)
df.head()

In [ ]:
PRODUCT_FACTORY_MAP = {
    "Wonka Bar - Nutty Crunch Surprise": ("Chocolate", "Lot\'s O\' Nuts", 32.881893, -111.768036),
    "Wonka Bar - Fudge Mallows": ("Chocolate", "Lot\'s O\' Nuts", 32.881893, -111.768036),
    "Wonka Bar -Scrumdiddlyumptious": ("Chocolate", "Lot\'s O\' Nuts", 32.881893, -111.768036),
    "Wonka Bar - Milk Chocolate": ("Chocolate", "Wicked Choccy\'s", 32.076176, -81.088371),
    "Wonka Bar - Triple Dazzle Caramel": ("Chocolate", "Wicked Choccy\'s", 32.076176, -81.088371),
    "Laffy Taffy": ("Sugar", "Sugar Shack", 48.11914, -96.18115),
    "SweeTARTS": ("Sugar", "Sugar Shack", 48.11914, -96.18115),
    "Nerds": ("Sugar", "Sugar Shack", 48.11914, -96.18115),
    "Fun Dip": ("Sugar", "Sugar Shack", 48.11914, -96.18115),
    "Fizzy Lifting Drinks": ("Other", "Sugar Shack", 48.11914, -96.18115),
    "Everlasting Gobstopper": ("Sugar", "Secret Factory", 41.446333, -90.565487),
    "Hair Toffee": ("Sugar", "The Other Factory", 35.1175, -89.971107),
    "Lickable Wallpaper": ("Other", "Secret Factory", 41.446333, -90.565487),
    "Wonka Gum": ("Other", "Secret Factory", 41.446333, -90.565487),
    "Kazookles": ("Other", "The Other Factory", 35.1175, -89.971107),
}


In [ ]:
df.columns = [c.strip() for c in df.columns]
for col in ["Order Date", "Ship Date"]:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
for col in ["Sales", "Units", "Gross Profit", "Cost"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["Ship Mode", "Country/Region", "City", "State/Province", "Division", "Region", "Product ID", "Product Name"]:
    df[col] = df[col].astype(str).str.strip()

df["Original Division"] = df["Division"]
df["Division"] = df["Product Name"].map(lambda x: PRODUCT_FACTORY_MAP.get(x, (np.nan, np.nan, np.nan, np.nan))[0]).fillna(df["Division"])
df["Factory"] = df["Product Name"].map(lambda x: PRODUCT_FACTORY_MAP.get(x, (np.nan, np.nan, np.nan, np.nan))[1])
df = df.dropna(subset=["Order Date", "Ship Date", "Sales", "Units", "Gross Profit", "Cost"])
df = df[(df["Sales"] > 0) & (df["Units"] > 0) & (df["Cost"] >= 0)].copy()
df["Gross Margin (%)"] = df["Gross Profit"] / df["Sales"] * 100
df["Profit per Unit"] = df["Gross Profit"] / df["Units"]
df["Cost to Sales (%)"] = df["Cost"] / df["Sales"] * 100
df.shape

In [ ]:
product = df.groupby(["Product ID", "Product Name", "Division", "Factory"], as_index=False).agg(
    Sales=("Sales", "sum"), Units=("Units", "sum"), Gross_Profit=("Gross Profit", "sum"), Cost=("Cost", "sum"), Orders=("Order ID", "nunique")
)
product["Gross Margin (%)"] = product["Gross_Profit"] / product["Sales"] * 100
product["Profit per Unit"] = product["Gross_Profit"] / product["Units"]
product["Revenue Contribution (%)"] = product["Sales"] / product["Sales"].sum() * 100
product["Profit Contribution (%)"] = product["Gross_Profit"] / product["Gross_Profit"].sum() * 100
product.sort_values("Gross_Profit", ascending=False)

In [ ]:
division = df.groupby("Division", as_index=False).agg(Sales=("Sales", "sum"), Units=("Units", "sum"), Gross_Profit=("Gross Profit", "sum"), Cost=("Cost", "sum"), Products=("Product Name", "nunique"))
division["Gross Margin (%)"] = division["Gross_Profit"] / division["Sales"] * 100
division["Revenue Share (%)"] = division["Sales"] / division["Sales"].sum() * 100
division["Profit Share (%)"] = division["Gross_Profit"] / division["Gross_Profit"].sum() * 100
division

In [ ]:
pareto_profit = product.sort_values("Gross_Profit", ascending=False).copy()
pareto_profit["Profit Share (%)"] = pareto_profit["Gross_Profit"] / pareto_profit["Gross_Profit"].sum() * 100
pareto_profit["Cumulative Profit Share (%)"] = pareto_profit["Profit Share (%)"].cumsum()
pareto_profit[["Product Name", "Profit Share (%)", "Cumulative Profit Share (%)"]]